# 02 · Setup Scanner

This notebook demonstrates how to:
1. Load daily OHLCV data for the watchlist symbols
2. Run the PullbackSetupDetector across the universe
3. Run the BreakoutSetupDetector across the universe
4. Rank and review the highest-scored setups

In [ ]:
import sys, warnings
sys.path.insert(0, '../src')
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

from martin_quant.setups.pullback_setup import PullbackSetupDetector, PullbackConfig
from martin_quant.setups.breakout_setup import BreakoutSetupDetector, BreakoutConfig
from martin_quant.features.ema import add_ema_features

WATCHLIST_PATH = Path('../data/outputs/watchlist.csv')
OHLCV_DIR     = Path('../data/outputs/daily/')
print('Paths ready')

## Load Watchlist & OHLCV Data

In [ ]:
watchlist = pd.read_csv(WATCHLIST_PATH) if WATCHLIST_PATH.exists() else pd.DataFrame()
symbols = watchlist['symbol'].tolist() if not watchlist.empty else []
print(f'Watchlist symbols: {len(symbols)}')

ohlcv_map = {}
for sym in symbols:
    path = OHLCV_DIR / f'{sym}_daily.parquet'
    if path.exists():
        df = pd.read_parquet(path)
        df['timestamp'] = pd.to_datetime(df['timestamp'])
        ohlcv_map[sym] = df
print(f'Loaded OHLCV for {len(ohlcv_map)} symbols')

## Scan Pullback Setups

In [ ]:
pullback_cfg = PullbackConfig(
    min_pullback_depth_pct=5.0,
    max_pullback_depth_pct=25.0,
    max_support_distance_pct=2.5,
    require_close_above_ema50=True,
    require_ema_stack=True,
)

pullback_detector = PullbackSetupDetector(config=pullback_cfg)
pullback_signals  = pullback_detector.scan_universe(symbols=list(ohlcv_map.keys()), ohlcv_map=ohlcv_map)
print(f'Pullback signals found: {len(pullback_signals)}')

pullback_df = pd.DataFrame([s.to_dict() for s in pullback_signals])
pullback_df.head(10) if not pullback_df.empty else print('No pullback setups today')

## Scan Breakout Setups

In [ ]:
breakout_cfg = BreakoutConfig(
    lookback_high_days=20,
    min_base_days=4,
    min_rvol_on_breakout=1.5,
)

breakout_detector = BreakoutSetupDetector(config=breakout_cfg)
breakout_signals  = breakout_detector.scan_universe(symbols=list(ohlcv_map.keys()), ohlcv_map=ohlcv_map)
print(f'Breakout signals found: {len(breakout_signals)}')

breakout_df = pd.DataFrame([s.to_dict() for s in breakout_signals])
breakout_df.head(10) if not breakout_df.empty else print('No breakout setups today')

## Score Distribution

In [ ]:
all_signals = pullback_df.assign(type='pullback') if not pullback_df.empty else pd.DataFrame()
if not breakout_df.empty:
    all_signals = pd.concat([all_signals, breakout_df.assign(type='breakout')], ignore_index=True)

if not all_signals.empty and 'score' in all_signals.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    for stype, grp in all_signals.groupby('type'):
        ax.hist(grp['score'], bins=15, alpha=0.7, label=stype, edgecolor='black')
    ax.set_xlabel('Setup Score')
    ax.set_ylabel('Count')
    ax.set_title('Setup Score Distribution by Type')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Chart Top Pullback Setup

In [ ]:
if pullback_signals:
    top = pullback_signals[0]
    df  = add_ema_features(ohlcv_map[top.symbol].tail(80), spans=(9, 20, 50))
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.plot(df['timestamp'], df['close'],  label='Close',  lw=1.5, color='black')
    ax.plot(df['timestamp'], df['ema_9'],  label='EMA 9',  lw=1,   color='blue',   ls='--')
    ax.plot(df['timestamp'], df['ema_20'], label='EMA 20', lw=1,   color='orange', ls='--')
    ax.plot(df['timestamp'], df['ema_50'], label='EMA 50', lw=1,   color='red',    ls='--')
    if top.support_level:
        ax.axhline(top.support_level, color='green', lw=1, ls=':', label='Support')
    if top.invalidation_level:
        ax.axhline(top.invalidation_level, color='red', lw=1, ls=':', label='Stop')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %d'))
    ax.set_title(f'{top.symbol} · Pullback Setup (score={top.score})')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()
    print(top.to_dict())